# 03 — Prompt-Invariant Comparison Sources

This notebook focuses on the four translation sources that do not vary by prompt: Wikipedia, Google Translate, EasyNMT, and Lingvanex. I sometimes use "baseline" in the code because these columns are prompt-invariant, but I do not treat all four sources as the same kind of evidence.

- **Wikipedia** is a community-curated reference. The values come from interlanguage links or page-title alignments attached to the `Digital Humanities` page. They are not fresh machine translations; they are labels that editors in particular language communities have accepted or stabilized.
- **Google Translate, EasyNMT, and Lingvanex** are machine-translation baselines. They are algorithmic outputs generated once per language, without the prompt variation used for the LLM services.

This distinction matters because these sources answer different questions. The MT baselines show how current translation systems handle DH terminology. Wikipedia shows whether a community-facing label already exists. LLM outputs sit between those poles and are handled separately in later notebooks.

I also keep Wikimedia language codes and Wikipedia translation strings conceptually separate. Notebook 01 uses Wikimedia as a language-code source. This notebook uses Wikipedia article labels as translation/reference data. The identifiers overlap, but the data layers are different.

Because these sources run once per language, they are useful for several checks before I compare prompt-sensitive LLM outputs:

- **Coverage**: which language families and scripts each source reaches.
- **Overlap**: which languages have multiple prompt-invariant sources and which have none.
- **Contribution**: whether a source verifies Google Translate or expands beyond it.
- **Fidelity**: whether an output translates the term or simply returns `Digital Humanities`.
- **Review signals**: which outputs trigger automated curation flags.
- **Filtering**: what remains under nominal, quality-filtered, and search-ready views.

The notebook loads manual `analysis_exclusion` decisions at the start, so most counts below are analysis-eligible languages rather than the full 881-row target set.


In [1]:
import os
import sys
from collections import defaultdict

import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family, detect_dominant_script
from scripts.exploration.explore_confidence_within_variant import load_variant_df
from scripts.exploration.translation_classifier import (
    curate_translation,
    has_source_leakage,
    is_placeholder_term,
    is_repetition_loop,
    has_extreme_term_length,
    has_unicode_escape,
)

DATA_DIR = get_data_directory_path()
TERM = "Digital Humanities"
TERM_SLUG = TERM.lower().replace(" ", "_")

BASELINE_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}

print(f"Data directory: {DATA_DIR}")

Retrieving translation pipeline data directory path...

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


In [2]:
# ── Manual exclusions ───────────────────────────────────────────────────────
from scripts.utils import load_manual_exclusions

excl_eval_dir = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(excl_eval_dir)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_langs)} language codes  (dropped from all analysis)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs  (excluded from search only)")
print(f"  term_correction    : {len(corrections)} corrections")

Manual exclusions loaded:
  analysis_exclusion : 95 language codes  (dropped from all analysis)
  search_exclusion   : 383 (language, term) pairs  (excluded from search only)
  term_correction    : 94 corrections


In [3]:
# Load the minimal-variant merged dataframe — baseline columns are identical across variants
df = load_variant_df(DATA_DIR, TERM_SLUG, "minimal")
df["language_family"] = df["language_code"].apply(get_language_family)

raw_total_langs = df["language_code"].nunique()
print(f"{TERM}: {raw_total_langs} languages loaded")

# Automated review signals from notebook 02
flags_path = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "automated_review_signals.csv")
flags = read_csv_file(flags_path)
print(f"Automated review signals: {len(flags)} rows, {flags.columns.tolist()}")

# Drop analysis-excluded languages
df = df[~df["language_code"].isin(analysis_langs)].reset_index(drop=True)
total_langs = df["language_code"].nunique()
print(f"After dropping {len(analysis_langs)} analysis-excluded languages: {total_langs} languages remain.")

Retrieving translation pipeline data directory path...

Digital Humanities: 881 languages loaded
Automated review signals: 881 rows, ['language_code', 'language_name', 'language_family', 'has_missing_rationale', 'missing_rationale_services', 'has_mixed_script', 'mixed_script_services', 'has_romanization', 'romanization_services', 'has_script_disagreement', 'script_disagr_services', 'has_source_term', 'source_term_services', 'has_placeholder_term', 'placeholder_term_services', 'has_repetition_loop', 'repetition_loop_services', 'has_extreme_term_length', 'extreme_term_length_services', 'has_unicode_escape', 'unicode_escape_services', 'has_short_translation', 'short_translation_services', 'has_any_mixing', 'any_mixing_services', 'has_refusal_rationale', 'refusal_rationale_services', 'has_transliteration_rationale', 'transliteration_rationale_services', 'has_placeholder_rationale', 'placeholder_rationale_services', 'has_language_name_term', 'language_name_term_services', 'has_unexpected_rationale_language', 'unexpected_rationale_language_servic

## 3.1 — Coverage by Source and Language Family

I begin by asking how many analysis-eligible languages each prompt-invariant source reaches. Because these sources do not use prompt variants, coverage is a single count per language. Wikipedia appears beside the MT systems for comparison, but I read it as a community reference, not as a translation service.


In [4]:
# Overall coverage counts
coverage_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    n = df[col].notna().sum()
    coverage_rows.append({
        "Service": svc,
        "Covered": int(n),
        "Missing": int(total_langs - n),
        "Coverage %": round(100 * n / total_langs, 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("Covered", ascending=False)
display(coverage_df)

,Service,Covered,Missing,Coverage %
1,Google Translate,229,557,29.1
3,Lingvanex,100,686,12.7
2,EasyNMT,92,694,11.7
0,Wikipedia,35,751,4.5


In [5]:
# Coverage bar chart
bar = alt.Chart(coverage_df).mark_bar().encode(
    x=alt.X("Covered:Q", title="Languages with a translation"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color("Service:N", legend=None),
    tooltip=["Service", "Covered", "Coverage %"],
).properties(title="Prompt-invariant source coverage", width=500, height=150)

rule = alt.Chart(pd.DataFrame({"x": [total_langs]})).mark_rule(
    color="grey", strokeDash=[4, 4]
).encode(x="x:Q")

(bar + rule)

alt.LayerChart(...)

In [6]:
# Coverage heatmap: service × language family
fam_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    grp = df.groupby("language_family")[col].agg(
        total="count",
        covered=lambda x: x.notna().sum(),
    ).reset_index()
    grp["Service"] = svc
    grp["pct"] = grp["covered"] / grp["total"]
    fam_rows.append(grp)

fam_df = pd.concat(fam_rows, ignore_index=True)

# Sort families by total language count (descending)
fam_totals = df.groupby("language_family")["language_code"].nunique().sort_values(ascending=False)
fam_order = fam_totals.index.tolist()

heatmap = alt.Chart(fam_df).mark_rect().encode(
    x=alt.X("Service:N", title=None),
    y=alt.Y("language_family:N", sort=fam_order, title=None),
    color=alt.Color(
        "pct:Q",
        scale=alt.Scale(scheme="greens", domain=[0, 1]),
        title="Coverage fraction",
    ),
    tooltip=[
        alt.Tooltip("Service:N"),
        alt.Tooltip("language_family:N", title="Family"),
        alt.Tooltip("covered:Q", title="Covered"),
        alt.Tooltip("total:Q", title="Total in family"),
        alt.Tooltip("pct:Q", title="Coverage", format=".0%"),
    ],
).properties(
    title="Prompt-invariant source coverage by language family",
    width=300,
    height=700,
)
heatmap

alt.Chart(...)

### Prompt-Invariant Word-Count Distribution

Word count is a simple but useful output profile. Since these sources do not vary by prompt, differences in length reflect source behavior rather than prompt design. I keep this view here and reserve the prompt-sensitive version for the LLM notebooks.


In [7]:
length_rows = []
for service, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    for wc in vals.str.split().str.len():
        length_rows.append({"Service": service, "Word Count": int(wc)})

length_df = pd.DataFrame(length_rows)

bars = alt.Chart(length_df).mark_bar(opacity=0.85).encode(
    x=alt.X("Word Count:Q", bin=alt.Bin(extent=[1, 12], step=1), title="word count"),
    y=alt.Y("count():Q", title="languages"),
    color=alt.Color("Service:N", scale=alt.Scale(scheme="tableau10"), legend=None),
    tooltip=["Service:N", alt.Tooltip("count():Q", title="languages")],
).properties(width=155, height=110)

median_rule = alt.Chart(length_df).mark_rule(color="red", strokeDash=[4, 2]).encode(
    x=alt.X("median(Word Count):Q"),
    tooltip=[alt.Tooltip("median(Word Count):Q", format=".1f", title="median word count")],
)

display(
    alt.layer(bars, median_rule, data=length_df).facet(
        facet=alt.Facet("Service:N", sort=list(BASELINE_SERVICES.keys()), title=None),
        columns=2,
        title=f"Prompt-invariant translation word-count distribution — {TERM} (red line = median)",
    )
)


alt.FacetChart(...)

## 3.2 — Source Overlap and Contribution

Coverage alone can hide whether a source adds new evidence or merely repeats another source's footprint. Here I count how many languages have zero, one, two, three, or all four prompt-invariant sources. I then ask what each source contributes: unique coverage, corroboration, source-term leakage, automated review signals, and search-ready retention.

I do not rank the sources on a single quality scale. A source can add coverage and still produce outputs that require review.


In [8]:
# Source overlap: how many prompt-invariant sources cover each language?
df["n_baseline"] = sum(
    df[col].notna().astype(int)
    for col in BASELINE_SERVICES.values()
    if col in df.columns
)

overlap_counts = df["n_baseline"].value_counts().sort_index().reset_index()
overlap_counts.columns = ["n_services", "n_languages"]
overlap_counts["label"] = overlap_counts["n_services"].map({
    0: "No prompt-invariant coverage",
    1: "1 source",
    2: "2 sources",
    3: "3 sources",
    4: "All 4 sources",
})

print(f"Prompt-invariant source overlap across {total_langs:,} analysis-eligible languages:")
for _, r in overlap_counts.iterrows():
    pct = r["n_languages"] / total_langs * 100
    print(f"  {r['label']:25s}: {r['n_languages']:4d}  ({pct:.1f}%)")

overlap_bar = alt.Chart(overlap_counts).mark_bar().encode(
    x=alt.X("label:N",
            sort=["No prompt-invariant coverage","1 source","2 sources","3 sources","All 4 sources"],
            title=None),
    y=alt.Y("n_languages:Q", title="languages"),
    color=alt.Color("n_services:O", scale=alt.Scale(scheme="blues"), legend=None),
    tooltip=["label:N", "n_languages:Q"],
).properties(width=340, height=200, title="How many prompt-invariant sources cover each language?")
display(overlap_bar)

zero_by_fam = (
    df[df["n_baseline"] == 0]
    .groupby("language_family")["language_code"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="n_zero")
)
print(f"\nZero prompt-invariant coverage languages by family:")
for _, r in zero_by_fam.iterrows():
    total_fam = df[df["language_family"] == r["language_family"]]["language_code"].nunique()
    pct = r["n_zero"] / total_fam * 100
    print(f"  {r['language_family']:45s}: {r['n_zero']:3d}/{total_fam:3d} ({pct:.0f}%)")


Prompt-invariant source overlap across 786 analysis-eligible languages:
  No prompt-invariant coverage:  544  (69.2%)
  1 source                 :  107  (13.6%)
  2 sources                :   78  (9.9%)
  3 sources                :   35  (4.5%)
  All 4 sources            :   22  (2.8%)


alt.Chart(...)


Zero prompt-invariant coverage languages by family:
  Indo-European languages                      : 142/227 (63%)
  Atlantic-Congo                               :  91/127 (72%)
  Austronesian languages                       :  42/ 77 (55%)
  Afro-Asiatic languages                       :  37/ 44 (84%)
  Sino-Tibetan languages                       :  36/ 44 (82%)
  Uralic languages                             :  19/ 26 (73%)
  Algic                                        :  17/ 17 (100%)
  Artificial languages                         :  12/ 13 (92%)
  Turkic                                       :   9/ 19 (47%)
  Austro-Asiatic languages                     :   9/ 12 (75%)
  Creoles and pidgins                          :   8/ 16 (50%)
  Tai-Kadai languages                          :   7/  8 (88%)
  Language isolate                             :   7/  8 (88%)
  Nakh-Daghestanian                            :   6/  8 (75%)
  Mande                                        :   6/  9 (67%)
 

### Pairwise Source Overlap — Verification vs Expansion

The overlap chart counts how many sources cover each language. The next question is which sources overlap with each other. I use Google Translate as the reference point because it is the broadest direct MT baseline in this dataset.

This lets me distinguish two kinds of contribution. If another source mostly overlaps with Google Translate, it may still be useful as corroboration. If it covers languages Google does not, it expands the available evidence. The matrix below reports pairwise overlap, then the chart separates each non-Google source into verification and expansion.


In [9]:
# Build per-service coverage sets (languages with a non-null translation)
cov_sets = {
    label: set(df.loc[df[col].notna(), 'language_code'])
    for label, col in BASELINE_SERVICES.items()
    if col in df.columns
}
svc_labels = list(cov_sets)

# Pairwise intersection matrix (counts)
overlap_mat = pd.DataFrame(
    [[len(cov_sets[a] & cov_sets[b]) for b in svc_labels] for a in svc_labels],
    index=svc_labels, columns=svc_labels,
)
print('── Pairwise overlap (number of languages covered by BOTH row and column) ──')
print(overlap_mat.to_string())

# % of ROW service's coverage that is also covered by COLUMN service
overlap_pct = pd.DataFrame(
    [[round(100 * len(cov_sets[a] & cov_sets[b]) / max(1, len(cov_sets[a])), 1)
      for b in svc_labels] for a in svc_labels],
    index=svc_labels, columns=svc_labels,
)
print('\n── % of ROW service\'s coverage that COLUMN service also covers ──')
print('(diagonal = 100% by definition; row "Wikipedia", column "GT" of 100% means every Wikipedia-covered language is also covered by GT)\n')
print(overlap_pct.to_string())

── Pairwise overlap (number of languages covered by BOTH row and column) ──
                  Wikipedia  Google Translate  EasyNMT  Lingvanex
Wikipedia                35                35       22         33
Google Translate         35               229       79        100
EasyNMT                  22                79       92         46
Lingvanex                33               100       46        100

── % of ROW service's coverage that COLUMN service also covers ──
(diagonal = 100% by definition; row "Wikipedia", column "GT" of 100% means every Wikipedia-covered language is also covered by GT)

                  Wikipedia  Google Translate  EasyNMT  Lingvanex
Wikipedia             100.0             100.0     62.9       94.3
Google Translate       15.3             100.0     34.5       43.7
EasyNMT                23.9              85.9    100.0       50.0
Lingvanex              33.0             100.0     46.0      100.0


In [10]:
# Direct verification-vs-expansion breakdown using Google Translate as the reference,
# since it has the broadest coverage (it's the de-facto MT baseline this paper benchmarks against).
if 'Google Translate' not in cov_sets:
    raise RuntimeError('Google Translate coverage set missing — check BASELINE_SERVICES.')
gt = cov_sets['Google Translate']

ve_rows = []
for svc in svc_labels:
    if svc == 'Google Translate': continue
    s = cov_sets[svc]
    overlap = s & gt
    only_this = s - gt
    ve_rows.append({
        'service': svc,
        'total_covered': len(s),
        'shared_with_GT': len(overlap),
        'unique_vs_GT':   len(only_this),
        'verification_pct': round(100 * len(overlap) / max(1, len(s)), 1),
        'expansion_pct':    round(100 * len(only_this) / max(1, len(s)), 1),
    })
ve = pd.DataFrame(ve_rows)
print('── Verification vs Expansion (Google Translate as reference) ──\n')
print(ve.to_string(index=False))

# Also: how many languages are uniquely covered by ONE service (no other baseline covers them)?
uniq_rows = []
for label in svc_labels:
    others = set().union(*(cov_sets[k] for k in svc_labels if k != label))
    uniq = cov_sets[label] - others
    uniq_rows.append({
        'service': label,
        'total_covered': len(cov_sets[label]),
        'unique_to_this_service': len(uniq),
        'unique_pct': round(100 * len(uniq) / max(1, len(cov_sets[label])), 1),
    })
uniq_df = pd.DataFrame(uniq_rows).sort_values('unique_to_this_service', ascending=False)
print('\n── Languages uniquely covered by exactly one prompt-invariant source ──')
print('(How much coverage *only* this source provides — covered by no other prompt-invariant source)\n')
print(uniq_df.to_string(index=False))

── Verification vs Expansion (Google Translate as reference) ──

  service  total_covered  shared_with_GT  unique_vs_GT  verification_pct  expansion_pct
Wikipedia             35              35             0             100.0            0.0
  EasyNMT             92              79            13              85.9           14.1
Lingvanex            100             100             0             100.0            0.0

── Languages uniquely covered by exactly one prompt-invariant source ──
(How much coverage *only* this source provides — covered by no other prompt-invariant source)

         service  total_covered  unique_to_this_service  unique_pct
Google Translate            229                      94        41.0
         EasyNMT             92                      13        14.1
       Wikipedia             35                       0         0.0
       Lingvanex            100                       0         0.0


In [11]:
# Visualisation: stacked bar showing verification vs expansion per non-GT service
ve_long = ve.melt(
    id_vars='service',
    value_vars=['shared_with_GT', 'unique_vs_GT'],
    var_name='kind', value_name='n_languages',
)
ve_long['kind'] = ve_long['kind'].map({
    'shared_with_GT': 'Shared with Google Translate (verification)',
    'unique_vs_GT':   'Not in Google Translate (expansion)',
})

bar = alt.Chart(ve_long).mark_bar().encode(
    y=alt.Y('service:N', sort=alt.SortField('n_languages', order='descending'),
            title=None),
    x=alt.X('n_languages:Q', title='languages covered'),
    color=alt.Color('kind:N',
                    scale=alt.Scale(
                        domain=['Shared with Google Translate (verification)',
                                'Not in Google Translate (expansion)'],
                        range=['#9ecae1', '#1976d2']),
                    legend=alt.Legend(orient='bottom', title=None)),
    tooltip=['service:N', 'kind:N', 'n_languages:Q'],
).properties(
    width=440, height=160,
    title=alt.Title(
        'How much does each non-Google baseline expand vs verify GT coverage?',
        subtitle='Dark = languages this service adds beyond GT; light = languages GT already covers',
    ),
)
labels = bar.mark_text(align='center', dx=0, fontSize=10, color='white').encode(
    text='n_languages:Q',
)
(bar + labels)

alt.LayerChart(...)

### Source Contribution and Signal Synthesis

This table gathers the contribution measures in one place. I use it to keep several facts visible at once: total coverage, unique coverage, overlap with Wikipedia, source-term leakage, automated review signals, and retention after filtering. The point is synthesis, not ranking. A source can be valuable because it verifies another source, expands coverage, or exposes a failure mode that matters methodologically.


In [12]:
import re as _re
import unicodedata as _unicodedata

def _norm_for_alignment(val):
    if not isinstance(val, str):
        return ""
    val = _unicodedata.normalize("NFKC", val).strip().casefold()
    val = _re.sub(r"\s+", " ", val)
    return val.strip(" .,:;!?¡¿()[]{}\"'`“”‘’/")

def _quality_failure(val):
    _, action = curate_translation(val)
    return (
        action in ("nulled", "placeholder")
        or is_repetition_loop(val)
        or has_extreme_term_length(val)
        or has_unicode_escape(val)
    )

def _review_signal_count(vals):
    total = 0
    for val in vals:
        _, action = curate_translation(val)
        total += int(action in ("nulled", "stripped", "placeholder"))
        total += int(is_repetition_loop(val))
        total += int(has_extreme_term_length(val))
        total += int(has_unicode_escape(val))
    return total

wiki_col = BASELINE_SERVICES.get("Wikipedia")
wiki_lookup = {}
if wiki_col and wiki_col in df.columns:
    wiki_lookup = {
        row["language_code"]: _norm_for_alignment(row[wiki_col])
        for _, row in df[df[wiki_col].notna()][["language_code", wiki_col]].iterrows()
    }

summary_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue

    covered_df = df[df[col].notna()][["language_code", col]].copy()
    covered_set = set(covered_df["language_code"])
    covered = len(covered_set)
    vals = covered_df[col].astype(str)

    other_sets = [cov_sets[k] for k in cov_sets if k != svc]
    unique_to_source = len(covered_set - set().union(*other_sets)) if other_sets else covered
    source_leakage = int(vals.apply(lambda v: has_source_leakage(v, TERM)).sum())
    exact_passthrough = int((vals.str.strip().str.casefold() == TERM.casefold()).sum())
    review_signal_hits = _review_signal_count(vals)
    quality_filtered_usable = int((~vals.apply(_quality_failure)).sum())
    search_ready_usable = int((~vals.apply(lambda v: _quality_failure(v) or has_source_leakage(v, TERM))).sum())

    wiki_match_rate = None
    if svc != "Wikipedia" and wiki_lookup:
        comparable = covered_df[covered_df["language_code"].isin(wiki_lookup)]
        if len(comparable):
            matches = comparable.apply(
                lambda row: _norm_for_alignment(row[col]) == wiki_lookup.get(row["language_code"], ""),
                axis=1,
            )
            wiki_match_rate = round(100 * matches.mean(), 1)

    summary_rows.append({
        "source": svc,
        "source_type": "community reference" if svc == "Wikipedia" else "MT baseline",
        "nominal_coverage": covered,
        "unique_coverage": unique_to_source,
        "source_leakage": source_leakage,
        "exact_passthrough": exact_passthrough,
        "automated_review_signal_hits": review_signal_hits,
        "quality_filtered_usable": quality_filtered_usable,
        "search_ready_usable": search_ready_usable,
        "wiki_exact_match_rate_pct": wiki_match_rate,
    })

source_signal_summary = pd.DataFrame(summary_rows).sort_values("nominal_coverage", ascending=False)
source_signal_summary["source_leakage_rate_pct"] = (
    source_signal_summary["source_leakage"] / source_signal_summary["nominal_coverage"] * 100
).round(1)
source_signal_summary["search_ready_retention_pct"] = (
    source_signal_summary["search_ready_usable"] / source_signal_summary["nominal_coverage"] * 100
).round(1)

display(source_signal_summary)

synth_long = source_signal_summary.melt(
    id_vars=["source", "source_type"],
    value_vars=[
        "nominal_coverage", "unique_coverage", "quality_filtered_usable",
        "search_ready_usable", "source_leakage", "automated_review_signal_hits",
    ],
    var_name="signal",
    value_name="n_languages",
)
signal_order = [
    "nominal_coverage", "unique_coverage", "quality_filtered_usable",
    "search_ready_usable", "source_leakage", "automated_review_signal_hits",
]
chart = alt.Chart(synth_long).mark_bar().encode(
    x=alt.X("n_languages:Q", title="languages / signal hits"),
    y=alt.Y("source:N", sort=alt.SortField("n_languages", order="descending"), title=None),
    color=alt.Color("source_type:N", title="source type"),
    row=alt.Row("signal:N", sort=signal_order, title=None),
    tooltip=["source:N", "source_type:N", "signal:N", "n_languages:Q"],
).properties(width=430, height=65, title="Prompt-invariant source contribution and review signals")
display(chart)


,source,source_type,nominal_coverage,unique_coverage,source_leakage,exact_passthrough,automated_review_signal_hits,quality_filtered_usable,search_ready_usable,wiki_exact_match_rate_pct,source_leakage_rate_pct,search_ready_retention_pct
1,Google Translate,MT baseline,229,94,17,14,0,229,212,62.9,7.4,92.6
3,Lingvanex,MT baseline,100,0,17,17,0,100,83,57.6,17.0,83.0
2,EasyNMT,MT baseline,92,13,1,1,3,89,88,31.8,1.1,95.7
0,Wikipedia,community reference,35,0,2,2,0,35,33,NaN,5.7,94.3


alt.Chart(...)

## 3.3 — Source-Specific Coverage Case Studies

The aggregate coverage charts do not explain why each source behaves the way it does. This section looks at three source-specific mechanisms: EasyNMT's model-availability gaps, Google's advertised support versus actual collected output, and Lingvanex's source-leakage pattern.

### EasyNMT Structural Gaps by Family

EasyNMT coverage is shaped by Helsinki-NLP OPUS-MT model availability. When no model exists for a source-target pair, the pipeline gets no output. That makes its gaps structural rather than random: entire language families can be out of scope because the necessary model was never available.

The chart below calculates EasyNMT coverage by reviewed family label. I use it to see where EasyNMT genuinely expands the comparison set and where it simply cannot participate.


In [13]:
# EasyNMT structural gaps by language family
enmt_col = BASELINE_SERVICES["EasyNMT"]

fam_enmt = (
    df.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_covered=(enmt_col, lambda x: x.notna().sum()),
    )
    .reset_index()
)
fam_enmt["coverage_rate"] = fam_enmt["n_covered"] / fam_enmt["n_langs"]
fam_enmt["zero_coverage"] = fam_enmt["n_covered"] == 0
fam_enmt = fam_enmt.sort_values("coverage_rate")

enmt_bar = alt.Chart(fam_enmt).mark_bar().encode(
    y=alt.Y("language_family:N", sort="x", title=None),
    x=alt.X("coverage_rate:Q", axis=alt.Axis(format="%"), title="EasyNMT coverage rate"),
    color=alt.condition(
        alt.datum.zero_coverage,
        alt.value("#d62728"),
        alt.value("#1f77b4"),
    ),
    tooltip=[
        "language_family:N",
        alt.Tooltip("coverage_rate:Q", format=".1%"),
        alt.Tooltip("n_covered:Q", title="languages covered"),
        alt.Tooltip("n_langs:Q", title="total languages"),
    ],
).properties(width=420, height=600,
             title="EasyNMT coverage by family — red = zero (no opus-mt model for this family)")
display(enmt_bar)

zero_fams = fam_enmt[fam_enmt["zero_coverage"]]["language_family"].tolist()
print(f"Families with zero EasyNMT coverage ({len(zero_fams)}):")
for f in zero_fams:
    n = int(fam_enmt.loc[fam_enmt["language_family"]==f, "n_langs"].iloc[0])
    print(f"  {f} ({n} languages)")


alt.Chart(...)

Families with zero EasyNMT coverage (52):
  Abkhaz-Adyge (4 languages)
  Uto-Aztecan (6 languages)
  Khoe-Kwadi (1 languages)
  Kru (1 languages)
  Language isolate (8 languages)
  Maban (1 languages)
  Mande (9 languages)
  Mayan (2 languages)
  Mongolic-Khitan (2 languages)
  Muskogean (4 languages)
  Nakh-Daghestanian (8 languages)
  Nilo-Saharan languages (3 languages)
  North American Indian languages (5 languages)
  Nubian (1 languages)
  Japonic languages (1 languages)
  Nuclear-Macro-Je (1 languages)
  Pama-Nyungan (1 languages)
  Quechuan (1 languages)
  Saharan (2 languages)
  Salishan (4 languages)
  Sign languages (1 languages)
  Siouan (3 languages)
  Songhay (4 languages)
  South American Indian languages (4 languages)
  Tai-Kadai (1 languages)
  Tai-Kadai languages (8 languages)
  Tungusic (2 languages)
  Tupian (1 languages)
  Turkic (19 languages)
  Otomanguean (1 languages)
  Iroquoian (4 languages)
  Kartvelian (3 languages)
  Indo-European (1 languages)
  Afro-Asiat

### EasyNMT Unique Coverage: What Does It Actually Add?

EasyNMT's unique coverage needs inspection because nominal coverage can overstate what is usable. The list below shows languages where EasyNMT is the only prompt-invariant source with an output for `Digital Humanities`.

Several patterns matter:

- Some outputs reflect the training distribution of OPUS-MT models, including models trained on religious parallel corpora such as JW.org.
- Some outputs are not usable direct translations even though they fill the field.
- Some outputs may be genuine translations and should not be rejected automatically.

I read EasyNMT as a source that can expand coverage, but whose interpretable contribution becomes smaller after automated signals and manual review.


In [14]:
# Inspect the EasyNMT-unique languages individually
gt_set    = cov_sets['Google Translate']
wiki_set  = cov_sets.get('Wikipedia', set())
lingv_set = cov_sets.get('Lingvanex', set())
_others    = gt_set | wiki_set | lingv_set
only_enmt = cov_sets['EasyNMT'] - _others

enmt_unique_df = (
    df[df['language_code'].isin(only_enmt)]
    [['language_code', 'language_name', 'language_family', 'enmt_translated_term']]
    .sort_values('language_name')
    .rename(columns={'enmt_translated_term': 'EasyNMT output for Digital Humanities'})
)
print(f'Languages where EasyNMT is the ONLY prompt-invariant source with coverage: {len(enmt_unique_df)}\n')
# Truncate long outputs for display
disp = enmt_unique_df.copy()
disp['EasyNMT output for Digital Humanities'] = disp['EasyNMT output for Digital Humanities'].astype(str).str.slice(0, 70)
print(disp.to_string(index=False))

Languages where EasyNMT is the ONLY prompt-invariant source with coverage: 13

language_code      language_name        language_family                                  EasyNMT output for Digital Humanities
           bi            Bislama    Creoles and pidgins                                       Ol Fasin Blong Man We Oli No Isi
          efi               Efik         Atlantic-Congo                                                 Digital Obio Ubọn̄ Owo
          gil         Gilbertese Austronesian languages                        Aroia Aomata n Tokanikai i Aoni Bwaai ni Kabane
           kj Kwanyama, Kuanyama         Atlantic-Congo                               Kala u na etaleko liwa li na sha novanhu
          loz               Lozi         Atlantic-Congo Litaba za Kwaikale ze Bulezwi Mwa Bibele ka za Litaba za Kwaikale ze B
          lun              Lunda         Atlantic-Congo                                       Kaanenu Yuma Yikamwekana Kumbidi
          mos              Mossi

### Google Official Support vs Actual Collection

The earlier charts show what this project actually collected from Google Translate. Google also publishes official support lists for its standard NMT endpoint and its Translation LLM endpoint. Cross-checking those lists against collected output lets me separate advertised support from delivered output.

I ask two questions:

1. For languages Google officially supports, did the pipeline collect a real translation?
2. For languages Google does not officially support, did the API still return something that looks like a translation?

The second question matters because unsupported does not always mean refused. APIs may return a source-term echo, a transliteration, a near-language calque, or a plausible-looking string. In the cells below, I count a Google output as a real translation only if it is non-empty, not an exact pass-through of `Digital Humanities`, not source-term leakage, and not a placeholder such as `Translation not available`.


In [15]:
# Load official-support columns from the metadata table and join to the working df
meta = pd.read_csv(
    os.path.join(DATA_DIR, 'metadata_files', 'language_codes_comprehensive.csv'),
    converters={'language_code': str},
)
support_cols = ['google_nmt_supported', 'google_translation_llm_supported']
assert all(c in meta.columns for c in support_cols), (
    'Run scripts/experiment/generate_language_codes.py or apply '
    'add_service_language_codes() to populate the support columns.'
)

df_sup = df.merge(meta[['language_code'] + support_cols], on='language_code', how='left')
for c in support_cols:
    df_sup[c] = df_sup[c].fillna(False).astype(bool)

print(f'Pipeline languages: {len(df_sup)}')
print(f'  Google NMT officially supported       : {df_sup["google_nmt_supported"].sum()}')
print(f'  Google Translation LLM officially supp.: {df_sup["google_translation_llm_supported"].sum()}')

Pipeline languages: 786
  Google NMT officially supported       : 172
  Google Translation LLM officially supp.: 80


In [16]:
# Classify each Google Translate output into 'real translation' or 'not real'
PLACEHOLDER_GT = {'translation not available', 'no direct translation', 'untranslatable',
                   'no translation', 'not available', 'unknown'}

def is_real_translation(val):
    if not isinstance(val, str): return False
    v = val.strip()
    if not v or v.lower() in {'nan', 'none', ''}: return False
    if v.strip().lower() == TERM.lower(): return False                # exact pass-through
    if has_source_leakage(v, TERM): return False                      # source-term leakage
    if v.strip().lower() in PLACEHOLDER_GT: return False             # placeholder
    if is_placeholder_term(v): return False                           # placeholder regex from classifier
    return True

df_sup['gt_real_translation'] = df_sup['gt_translated_term'].apply(is_real_translation)

# Build the 2×2 quadrant for Google NMT
def _quadrant(df_in, support_col, real_col):
    return pd.DataFrame({
        'support': ['Officially supported', 'Officially supported',
                    'NOT officially supported', 'NOT officially supported'],
        'outcome': ['Got a real translation', 'Got nothing/junk',
                    'Got a real translation', 'Got nothing/junk'],
        'count': [
            int(( df_in[support_col] &  df_in[real_col]).sum()),
            int(( df_in[support_col] & ~df_in[real_col]).sum()),
            int((~df_in[support_col] &  df_in[real_col]).sum()),
            int((~df_in[support_col] & ~df_in[real_col]).sum()),
        ],
    })

nmt_quadrant = _quadrant(df_sup, 'google_nmt_supported', 'gt_real_translation')
nmt_quadrant['pct'] = (nmt_quadrant['count'] / len(df_sup) * 100).round(1)
print('── Google NMT: officially supported × actually delivered ──')
print(nmt_quadrant.to_string(index=False))

── Google NMT: officially supported × actually delivered ──
                 support                outcome  count  pct
    Officially supported Got a real translation    159 20.2
    Officially supported       Got nothing/junk     13  1.7
NOT officially supported Got a real translation     53  6.7
NOT officially supported       Got nothing/junk    561 71.4


In [17]:
# Visualise the 2×2 as a heatmap
quadrant_labels = {
    ('Officially supported', 'Got a real translation'):     '✓ advertised + delivered',
    ('Officially supported', 'Got nothing/junk'):            '✗ advertised but failed',
    ('NOT officially supported', 'Got a real translation'):  '⊕ delivered without support',
    ('NOT officially supported', 'Got nothing/junk'):        '∅ expected absence',
}
nmt_quadrant['label'] = nmt_quadrant.apply(
    lambda r: quadrant_labels.get((r['support'], r['outcome']), ''), axis=1,
)
nmt_quadrant['display'] = nmt_quadrant.apply(
    lambda r: f"{r['count']} ({r['pct']:.1f}%)", axis=1,
)

heat = alt.Chart(nmt_quadrant).mark_rect().encode(
    x=alt.X('outcome:N', sort=['Got a real translation', 'Got nothing/junk'],
            title='Actual collection outcome',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('support:N', sort=['Officially supported', 'NOT officially supported'],
            title='Google NMT advertised support'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['support:N', 'outcome:N', 'count:Q', 'pct:Q', 'label:N'],
).properties(width=320, height=180,
             title=alt.Title('Google NMT: advertised vs delivered',
                             subtitle='Top-left = ideal; top-right = silent failures; bottom-left = implicit coverage'))
labels = heat.mark_text(fontSize=12, fontWeight='bold').encode(
    text='display:N',
    color=alt.condition('datum.count > 400', alt.value('white'), alt.value('black')),
)
(heat + labels)

alt.LayerChart(...)

In [18]:
# Same quadrant for Google's Translation LLM endpoint
llm_quadrant = _quadrant(df_sup, 'google_translation_llm_supported', 'gt_real_translation')
llm_quadrant['pct'] = (llm_quadrant['count'] / len(df_sup) * 100).round(1)
print('── Google Translation LLM: officially supported × actually delivered ──')
print(llm_quadrant.to_string(index=False))

── Google Translation LLM: officially supported × actually delivered ──
                 support                outcome  count  pct
    Officially supported Got a real translation     75  9.5
    Officially supported       Got nothing/junk      5  0.6
NOT officially supported Got a real translation    137 17.4
NOT officially supported       Got nothing/junk    569 72.4


In [19]:
# Spot-check samples for the two unexpected buckets
advertised_failed = df_sup[df_sup['google_nmt_supported'] & ~df_sup['gt_real_translation']]
implicit_coverage = df_sup[~df_sup['google_nmt_supported'] &  df_sup['gt_real_translation']]

print(f'── Advertised but failed: {len(advertised_failed)} languages ──')
print('(Google officially supports these in NMT but our pipeline did not collect a real translation)\n')
if len(advertised_failed):
    print(advertised_failed[['language_code', 'language_name', 'gt_translated_term']]
          .head(15).to_string(index=False))

print(f'\n── Implicit coverage: {len(implicit_coverage)} languages ──')
print('(Not in any official Google NMT support list, but the API returned a real-looking translation anyway)\n')
if len(implicit_coverage):
    print(implicit_coverage[['language_code', 'language_name', 'gt_translated_term']]
          .head(15).to_string(index=False))

── Advertised but failed: 13 languages ──
(Google officially supports these in NMT but our pipeline did not collect a real translation)

language_code language_name                      gt_translated_term
           nr South Ndebele                    I-Digital Humanities
          pag    Pangasinan                      Digital Humanities
          pam   Kapampangan                      Digital Humanities
      map-bms    Banyumasan                                     NaN
          lus          Mizo Digital Humanities hmanga thil tih a ni
           tl       Tagalog                      Digital Humanities
          ach         Acoli                      Digital Humanities
          bik         Bikol                      Digital Humanities
          fil      Filipino                      Digital Humanities
           lb Luxembourgish                      Digital Humanities
          hil    Hiligaynon                      Digital Humanities
           zu          Zulu                    

### Lingvanex Source-Leakage Drilldown

Lingvanex has enough nominal coverage to matter, but the search-ready funnel removes many outputs that preserve the English source term. I do not treat every English phrase as automatically wrong; borrowed English terminology can be conventional. The narrower question here is where Lingvanex preserves `Digital Humanities` or `DH`, and whether those cases cluster by family or script.


In [20]:
ling_col = BASELINE_SERVICES.get("Lingvanex")
if not ling_col or ling_col not in df.columns:
    print("Lingvanex column not available.")
else:
    ling_rows = df[df[ling_col].notna()][
        ["language_code", "language_name", "language_family", ling_col]
    ].copy()
    ling_rows = ling_rows.rename(columns={ling_col: "lingvanex_term"})
    ling_rows["source_leakage"] = ling_rows["lingvanex_term"].apply(lambda v: has_source_leakage(v, TERM))
    ling_rows["exact_passthrough"] = ling_rows["lingvanex_term"].str.strip().str.casefold() == TERM.casefold()
    ling_rows["dominant_script"] = ling_rows["lingvanex_term"].apply(detect_dominant_script)

    total_ling = len(ling_rows)
    leak_ling = int(ling_rows["source_leakage"].sum())
    exact_ling = int(ling_rows["exact_passthrough"].sum())
    print(f"Lingvanex translated {total_ling} analysis-eligible languages.")
    print(f"  Source-term leakage: {leak_ling} ({leak_ling / total_ling * 100:.1f}%)")
    print(f"  Exact pass-through : {exact_ling} ({exact_ling / total_ling * 100:.1f}%)")

    ling_leak = ling_rows[ling_rows["source_leakage"]].copy()
    if ling_leak.empty:
        print("No Lingvanex source-leakage cases found.")
    else:
        fam_summary = (
            ling_rows.groupby("language_family", as_index=False)
            .agg(
                n_lingvanex=("language_code", "nunique"),
                n_source_leakage=("source_leakage", "sum"),
                n_exact_passthrough=("exact_passthrough", "sum"),
            )
        )
        fam_summary["source_leakage_rate"] = (
            fam_summary["n_source_leakage"] / fam_summary["n_lingvanex"] * 100
        ).round(1)
        fam_summary = fam_summary[fam_summary["n_source_leakage"] > 0].sort_values(
            ["n_source_leakage", "source_leakage_rate"], ascending=[False, False]
        )
        print("\nLingvanex source leakage by family:")
        display(fam_summary)

        script_summary = (
            ling_rows.groupby("dominant_script", as_index=False)
            .agg(
                n_lingvanex=("language_code", "nunique"),
                n_source_leakage=("source_leakage", "sum"),
                n_exact_passthrough=("exact_passthrough", "sum"),
            )
        )
        script_summary["source_leakage_rate"] = (
            script_summary["n_source_leakage"] / script_summary["n_lingvanex"] * 100
        ).round(1)
        print("\nLingvanex source leakage by dominant script:")
        display(script_summary.sort_values(["n_source_leakage", "source_leakage_rate"], ascending=[False, False]))

        print("\nLingvanex source-leakage cases:")
        display(
            ling_leak[["language_code", "language_name", "language_family", "dominant_script", "exact_passthrough", "lingvanex_term"]]
            .sort_values(["exact_passthrough", "language_family", "language_name"], ascending=[False, True, True])
            .reset_index(drop=True)
        )


Lingvanex translated 100 analysis-eligible languages.
  Source-term leakage: 17 (17.0%)
  Exact pass-through : 17 (17.0%)

Lingvanex source leakage by family:


,language_family,n_lingvanex,n_source_leakage,n_exact_passthrough,source_leakage_rate
5,Atlantic-Congo,8,6,6,75.0
11,Indo-European languages,52,6,6,11.5
7,Austronesian languages,8,3,3,37.5
10,Hmong-Mien languages,1,1,1,100.0
16,Tai-Kadai languages,1,1,1,100.0



Lingvanex source leakage by dominant script:


,dominant_script,n_lingvanex,n_source_leakage,n_exact_passthrough,source_leakage_rate
15,Latin,62,17,17,27.4
0,Arabic,6,0,0,0.0
1,Armenian,1,0,0,0.0
2,Bengali,1,0,0,0.0
3,CJK,1,0,0,0.0
4,Cyrillic,11,0,0,0.0
5,Devanagari,3,0,0,0.0
6,Ethiopic,1,0,0,0.0
7,Georgian,1,0,0,0.0
8,Greek,1,0,0,0.0



Lingvanex source-leakage cases:


,language_code,language_name,language_family,dominant_script,exact_passthrough,lingvanex_term
0,ny,Chichewa; Chewa; Nyanja,Atlantic-Congo,Latin,True,Digital Humanities
1,ig,Igbo,Atlantic-Congo,Latin,True,Digital Humanities
2,sn,Shona,Atlantic-Congo,Latin,True,Digital Humanities
3,st,Southern Sotho,Atlantic-Congo,Latin,True,Digital Humanities
4,yo,Yoruba,Atlantic-Congo,Latin,True,Digital Humanities
5,zu,Zulu,Atlantic-Congo,Latin,True,Digital Humanities
6,haw,Hawaiian,Austronesian languages,Latin,True,Digital Humanities
7,mg,Malagasy,Austronesian languages,Latin,True,Digital Humanities
8,tl,Tagalog,Austronesian languages,Latin,True,Digital Humanities
9,hmn,Hmong,Hmong-Mien languages,Latin,True,Digital Humanities


## 3.4 — Wikipedia and Community Alignment

I do not treat Wikipedia labels as authoritative translations. I use them as community-facing evidence: page titles or interlanguage labels that have been edited into place. Exact agreement between Wikipedia and an MT source can mean several things: a stable community term, a borrowed English phrase, a shared calque, or a source-term echo. This section therefore describes alignment rather than declaring agreement to be correctness.


In [21]:
import re
import unicodedata

MT_SOURCE_COLS = {
    "Google Translate": BASELINE_SERVICES["Google Translate"],
    "EasyNMT": BASELINE_SERVICES["EasyNMT"],
    "Lingvanex": BASELINE_SERVICES["Lingvanex"],
}

def normalize_term(val):
    if not isinstance(val, str):
        return ""
    val = unicodedata.normalize("NFKC", val).strip().casefold()
    val = re.sub(r"\s+", " ", val)
    return val.strip(" .,:;!?¡¿()[]{}\"'`“”‘’/")

wiki_col = BASELINE_SERVICES["Wikipedia"]
wiki_rows = df[df[wiki_col].notna()].copy()
wiki_rows["wiki_norm"] = wiki_rows[wiki_col].apply(normalize_term)
wiki_rows["wiki_has_source_leakage"] = wiki_rows[wiki_col].apply(lambda v: has_source_leakage(v, TERM))

align_rows = []
for svc, col in MT_SOURCE_COLS.items():
    comparable = wiki_rows[wiki_rows[col].notna()].copy()
    if comparable.empty:
        align_rows.append({
            "service": svc,
            "wiki_languages_with_mt": 0,
            "exact_normalized_matches": 0,
            "match_rate": 0.0,
            "matches_where_wiki_leaks_source": 0,
        })
        continue
    mt_norm = comparable[col].apply(normalize_term)
    matches = mt_norm == comparable["wiki_norm"]
    align_rows.append({
        "service": svc,
        "wiki_languages_with_mt": len(comparable),
        "exact_normalized_matches": int(matches.sum()),
        "match_rate": round(100 * matches.mean(), 1),
        "matches_where_wiki_leaks_source": int((matches & comparable["wiki_has_source_leakage"]).sum()),
    })

wiki_alignment_df = pd.DataFrame(align_rows)
print(f"Wikipedia-covered languages in analysis set: {len(wiki_rows)}")
print(f"  Wikipedia labels with source-term leakage: {int(wiki_rows['wiki_has_source_leakage'].sum())}")
display(wiki_alignment_df)

wiki_match_detail_rows = []
for _, row in wiki_rows.iterrows():
    mt_matches = []
    mt_available = []
    for svc, col in MT_SOURCE_COLS.items():
        if pd.notna(row.get(col)):
            mt_available.append(svc)
            if normalize_term(row[col]) == row["wiki_norm"]:
                mt_matches.append(svc)
    wiki_match_detail_rows.append({
        "language_code": row["language_code"],
        "language_name": row["language_name"],
        "language_family": row["language_family"],
        "wikipedia_term": row[wiki_col],
        "wiki_has_source_leakage": row["wiki_has_source_leakage"],
        "n_mt_available": len(mt_available),
        "n_mt_exact_matches": len(mt_matches),
        "mt_exact_matches": "; ".join(mt_matches),
    })

wiki_match_detail = pd.DataFrame(wiki_match_detail_rows)
print("\nWikipedia labels with at least one exact normalized MT match:")
display(
    wiki_match_detail[wiki_match_detail["n_mt_exact_matches"] > 0]
    .sort_values(["wiki_has_source_leakage", "n_mt_exact_matches", "language_name"], ascending=[False, False, True])
    .reset_index(drop=True)
)

match_chart = alt.Chart(wiki_alignment_df).mark_bar().encode(
    x=alt.X("match_rate:Q", title="% of Wikipedia-covered languages with exact normalized match"),
    y=alt.Y("service:N", sort="-x", title=None),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N", "wiki_languages_with_mt:Q", "exact_normalized_matches:Q", "match_rate:Q", "matches_where_wiki_leaks_source:Q"],
).properties(width=430, height=130, title="MT alignment with Wikipedia labels (normalized exact match)")
display(match_chart)


Wikipedia-covered languages in analysis set: 35
  Wikipedia labels with source-term leakage: 2


,service,wiki_languages_with_mt,exact_normalized_matches,match_rate,matches_where_wiki_leaks_source
0,Google Translate,35,22,62.9,1
1,EasyNMT,22,7,31.8,0
2,Lingvanex,33,19,57.6,1



Wikipedia labels with at least one exact normalized MT match:


,language_code,language_name,language_family,wikipedia_term,wiki_has_source_leakage,n_mt_available,n_mt_exact_matches,mt_exact_matches
0,en,English,Indo-European languages,digital humanities,True,2,2,Google Translate; Lingvanex
1,ca,Catalan; Valencian,Indo-European languages,Humanitats digitals,False,3,3,Google Translate; EasyNMT; Lingvanex
2,fr,French,Indo-European languages,Humanités numériques,False,3,3,Google Translate; EasyNMT; Lingvanex
3,el,"Greek, Modern",Indo-European languages,Ψηφιακές Ανθρωπιστικές Επιστήμες,False,3,3,Google Translate; EasyNMT; Lingvanex
4,ru,Russian,Indo-European languages,Цифровые гуманитарные науки,False,3,3,Google Translate; EasyNMT; Lingvanex
5,es,Spanish; Castilian,Indo-European languages,Humanidades digitales,False,3,3,Google Translate; EasyNMT; Lingvanex
6,sv,Swedish,Indo-European languages,Digital humaniora,False,3,3,Google Translate; EasyNMT; Lingvanex
7,hy,Armenian,Armenian languages,Թվային հումանիտար գիտություններ,False,3,2,Google Translate; Lingvanex
8,az,Azerbaijani,Altaic languages,Rəqəmsal humanitar elmlər,False,3,2,Google Translate; Lingvanex
9,eu,Basque,Basque languages,Humanitate digitalak,False,3,2,Google Translate; Lingvanex


alt.Chart(...)

## 3.5 — Source-Term Pass-Through and Review Signals

A source that returns `Digital Humanities` unchanged has technically produced a non-null value, but it has not produced a useful translated search term. I keep these pass-throughs in the exploratory data because they reveal service behavior, then exclude them from the search-ready view later. This section measures source-term leakage before broadening to other automated review signals.


In [22]:
pass_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    exact = (vals.str.strip().str.lower() == TERM.lower()).sum()
    contains = vals.str.contains(TERM, case=False, na=False).sum()
    source_leak = sum(has_source_leakage(v, TERM) for v in vals)
    pass_rows.append({
        "Service": svc,
        "Covered": covered,
        "Exact pass-through": int(exact),
        "Contains source term": int(contains),
        "Source leakage (incl. initials)": int(source_leak),
        "Pass-through %": round(100 * exact / covered, 1) if covered else 0,
        "Source leakage %": round(100 * source_leak / covered, 1) if covered else 0,
    })

pass_df = pd.DataFrame(pass_rows)
display(pass_df)

,Service,Covered,Exact pass-through,Contains source term,Source leakage (incl. initials),Pass-through %,Source leakage %
0,Wikipedia,35,2,2,2,5.7,5.7
1,Google Translate,229,14,17,17,6.1,7.4
2,EasyNMT,92,1,1,1,1.1,1.1
3,Lingvanex,100,17,17,17,17.0,17.0


In [23]:
# Stacked bar: genuine translation vs pass-through vs source-leakage
stacked_rows = []
for _, row in pass_df.iterrows():
    exact = row["Exact pass-through"]
    leak_extra = row["Source leakage (incl. initials)"] - exact
    genuine = row["Covered"] - row["Source leakage (incl. initials)"]
    stacked_rows += [
        {"Service": row["Service"], "Category": "Genuine translation", "Count": genuine},
        {"Service": row["Service"], "Category": "Contains DH (non-exact)", "Count": max(0, leak_extra)},
        {"Service": row["Service"], "Category": "Exact pass-through", "Count": exact},
    ]

stacked_df = pd.DataFrame(stacked_rows)

stacked_bar = alt.Chart(stacked_df).mark_bar().encode(
    x=alt.X("sum(Count):Q", title="Languages"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color(
        "Category:N",
        scale=alt.Scale(
            domain=["Genuine translation", "Contains DH (non-exact)", "Exact pass-through"],
            range=["#4c9b5e", "#f0a830", "#c9413a"],
        ),
    ),
    tooltip=["Service", "Category", "sum(Count):Q"],
    order=alt.Order("Category:N", sort="ascending"),
).properties(title="Prompt-invariant source breakdown: genuine vs pass-through", width=500, height=150)

stacked_bar

alt.Chart(...)

In [24]:
# Which languages are pass-throughs? Show the top offenders per service
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    passthrough_mask = df[col].str.strip().str.lower() == TERM.lower()
    rows = df.loc[passthrough_mask, ["language_code", "language_name", "language_family", col]]
    if len(rows):
        print(f"\n{svc} — {len(rows)} exact pass-throughs:")
        display(rows.reset_index(drop=True))
    else:
        print(f"\n{svc} — no exact pass-throughs")


Wikipedia — 2 exact pass-throughs:


,language_code,language_name,language_family,wikipedia_translated_term
0,de,German,Indo-European languages,Digital Humanities
1,en,English,Indo-European languages,digital humanities



Google Translate — 14 exact pass-throughs:


,language_code,language_name,language_family,gt_translated_term
0,pag,Pangasinan,Austronesian languages,Digital Humanities
1,pam,Kapampangan,Austronesian languages,Digital Humanities
2,mh,Marshallese,Austronesian languages,Digital Humanities
3,tl,Tagalog,Austronesian languages,Digital Humanities
4,sus,Susu,Mande,Digital Humanities
5,ch,Chamorro,Austronesian languages,Digital Humanities
6,ach,Acoli,Nilotic,Digital Humanities
7,bik,Bikol,Austronesian languages,Digital Humanities
8,bcl,Bikol,Austronesian languages,Digital Humanities
9,fil,Filipino,Austronesian languages,Digital Humanities



EasyNMT — 1 exact pass-throughs:


,language_code,language_name,language_family,enmt_translated_term
0,it,Italian,Indo-European languages,Digital Humanities



Lingvanex — 17 exact pass-throughs:


,language_code,language_name,language_family,lingvanex_translated_term
0,no,Norwegian,Indo-European languages,Digital Humanities
1,ny,Chichewa; Chewa; Nyanja,Atlantic-Congo,Digital Humanities
2,nl,Dutch,Indo-European languages,Digital Humanities
3,mg,Malagasy,Austronesian languages,Digital Humanities
4,yo,Yoruba,Atlantic-Congo,Digital Humanities
5,sn,Shona,Atlantic-Congo,Digital Humanities
6,tl,Tagalog,Austronesian languages,Digital Humanities
7,st,Southern Sotho,Atlantic-Congo,Digital Humanities
8,lo,Lao,Tai-Kadai languages,Digital Humanities
9,bs,Bosnian,Indo-European languages,Digital Humanities


### Automated Review Signals per Prompt-Invariant Source

Here I run the prompt-invariant outputs through the translation-level checks in `translation_classifier.py`. These sources do not produce rationales and do not vary by prompt, so rationale-level and prompt-disagreement signals do not apply. I use this section to see which source outputs need review before manual decisions or downstream filtering.


In [25]:
CHECKERS = {
    "placeholder": is_placeholder_term,
    "repetition_loop": is_repetition_loop,
    "extreme_length": has_extreme_term_length,
    "unicode_escape": has_unicode_escape,
}

flag_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    row = {"Service": svc, "Covered": covered}
    for flag_name, fn in CHECKERS.items():
        count = sum(fn(v) for v in vals)
        row[flag_name] = count
        row[f"{flag_name}_pct"] = round(100 * count / covered, 1) if covered else 0

    # mixed-script / stripped via curate_translation
    nulled = stripped = 0
    for v in vals:
        _, action = curate_translation(v)
        if action == "nulled":
            nulled += 1
        elif action == "stripped":
            stripped += 1
    row["mixed_script"] = nulled
    row["mixed_script_pct"] = round(100 * nulled / covered, 1) if covered else 0
    row["romanization_stripped"] = stripped
    row["romanization_stripped_pct"] = round(100 * stripped / covered, 1) if covered else 0

    flag_rows.append(row)

flag_df = pd.DataFrame(flag_rows)

# Display count table
count_cols = ["Service", "Covered", "placeholder", "repetition_loop", "extreme_length",
              "unicode_escape", "mixed_script", "romanization_stripped"]
display(flag_df[count_cols])

,Service,Covered,placeholder,repetition_loop,extreme_length,unicode_escape,mixed_script,romanization_stripped
0,Wikipedia,35,0,0,0,0,0,0
1,Google Translate,229,0,0,0,0,0,0
2,EasyNMT,92,0,1,2,0,0,0
3,Lingvanex,100,0,0,0,0,0,0


In [26]:
# Flag rate chart
flag_long_rows = []
flag_display = {
    "placeholder": "Placeholder/refusal",
    "repetition_loop": "Repetition loop",
    "extreme_length": "Extreme length (>100 chars)",
    "unicode_escape": "Unicode escape (\\uXXXX)",
    "mixed_script": "Mixed script (nulled)",
    "romanization_stripped": "Romanization stripped",
}
for _, row in flag_df.iterrows():
    for flag, label in flag_display.items():
        pct_col = f"{flag}_pct"
        if pct_col in flag_df.columns:
            flag_long_rows.append({
                "Service": row["Service"],
                "Flag": label,
                "Rate": row[pct_col],
                "Count": int(row[flag]),
            })

flag_long_df = pd.DataFrame(flag_long_rows)

# Only show flags that actually fired
nonzero_flags = flag_long_df.groupby("Flag")["Count"].sum()
active_flags = nonzero_flags[nonzero_flags > 0].index.tolist()

if active_flags:
    active_df = flag_long_df[flag_long_df["Flag"].isin(active_flags)]
    flag_chart = alt.Chart(active_df).mark_bar().encode(
        x=alt.X("Rate:Q", title="% of covered translations"),
        y=alt.Y("Service:N", title=None),
        color=alt.Color("Service:N", legend=None),
        row=alt.Row("Flag:N", title=None),
        tooltip=["Service", "Flag", "Count", alt.Tooltip("Rate:Q", format=".1f", title="%")],
    ).properties(width=400, height=80, title="Prompt-invariant automated review signal rates (% of covered languages)")
    display(flag_chart)
else:
    print("No automated review signals fired for any prompt-invariant source.")

alt.Chart(...)

In [27]:
# Show offending rows for any flagged prompt-invariant translations
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    rows_out = []
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        flags_fired = []
        for flag_name, fn in CHECKERS.items():
            if fn(v):
                flags_fired.append(flag_name)
        _, action = curate_translation(v)
        if action in ("nulled", "stripped"):
            flags_fired.append(action)
        if flags_fired:
            rows_out.append({
                "language_code": row["language_code"],
                "language_name": row["language_name"],
                "language_family": row["language_family"],
                "translation": v[:120],
                "flags": ", ".join(flags_fired),
            })
    if rows_out:
        print(f"\n{svc} — {len(rows_out)} flagged translations:")
        display(pd.DataFrame(rows_out))
    else:
        print(f"\n{svc} — no automated review signals")


Wikipedia — no automated review signals

Google Translate — no automated review signals

EasyNMT — 3 flagged translations:


,language_code,language_name,language_family,translation,flags
0,loz,Lozi,Atlantic-Congo,Litaba za Kwaikale ze Bulezwi Mwa Bibele ka za...,extreme_length
1,vi,Vietnamese,Austro-Asiatic languages,Hệ bình bình bình bình bình bình bình bình bìn...,repetition_loop
2,bg,Bulgarian,Indo-European languages,(Средредредредредредредредредредредредредредре...,extreme_length



Lingvanex — no automated review signals


## 3.6 — Script and Curation Signals

Script behavior is one part of the review-signal story. This section asks which writing systems appear in prompt-invariant outputs and which translations require curation through `curate_translation()`. A script mismatch does not prove an output is wrong, but it is a strong signal for review, especially when a source falls back to Latin script or mixes scripts in ways that look like leakage or transliteration.


In [28]:
script_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        script = detect_dominant_script(v)
        _, action = curate_translation(v)
        script_rows.append({
            "Service": svc,
            "language_code": row["language_code"],
            "language_name": row["language_name"],
            "language_family": row["language_family"],
            "translation": v[:80],
            "dominant_script": script,
            "clean_action": action,
        })

script_df = pd.DataFrame(script_rows)

# Script distribution per service
script_dist = (
    script_df.groupby(["Service", "dominant_script"])
    .size()
    .reset_index(name="count")
)
script_chart = alt.Chart(script_dist).mark_bar().encode(
    x=alt.X("count:Q", title="Translations"),
    y=alt.Y("dominant_script:N", title=None, sort="-x"),
    color=alt.Color("Service:N"),
    tooltip=["Service", "dominant_script", "count"],
).properties(
    title="Script distribution of prompt-invariant translations",
    width=400, height=250
)
script_chart

alt.Chart(...)

In [29]:
# Translations that needed cleaning (stripped or nulled)
needing_clean = script_df[script_df["clean_action"].isin(["stripped", "nulled"])]
if len(needing_clean):
    print(f"{len(needing_clean)} prompt-invariant translations needed cleaning:")
    display(needing_clean[["Service", "language_name", "translation", "clean_action"]].reset_index(drop=True))
else:
    print("No prompt-invariant translations required cleaning — all pass through unchanged or as-is.")

No prompt-invariant translations required cleaning — all pass through unchanged or as-is.


## 3.7 — Filter Funnel: Full, Quality-Filtered, and Search-Ready Views

The final section turns the descriptive signals into increasingly strict views of coverage.

- **Nominal coverage** counts any non-null output.
- **Quality-filtered coverage** removes likely term failures: mixed-script nulls, placeholder/refusal terms, repetition loops, extreme length, and literal Unicode escapes.
- **Search-ready coverage** also removes source-term leakage, because exact `Digital Humanities` echoes are poor search terms even when they remain useful evidence of service behavior.

This funnel lets me keep two things true at once: pass-throughs remain meaningful evidence in exploratory analysis, but they should not be treated as usable translated search terms.


In [30]:
# Quality-filtered view: remove likely term failures
tier1_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier1_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (quality filters)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier1_df = pd.DataFrame(tier1_rows)
display(tier1_df)

,Service,Covered,Excluded (quality filters),Usable,Usable %
0,Wikipedia,35,0,35,100.0
1,Google Translate,229,0,229,100.0
2,EasyNMT,92,3,89,96.7
3,Lingvanex,100,0,100,100.0


In [31]:
# Search-ready view: additionally exclude pass-throughs (source term in translation)
tier2_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
            or has_source_leakage(v, TERM)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier2_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (quality + source leakage)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier2_df = pd.DataFrame(tier2_rows)
print("Search-ready view (quality filters + source leakage exclusion):")
display(tier2_df)

Search-ready view (quality filters + source leakage exclusion):


,Service,Covered,Excluded (quality + source leakage),Usable,Usable %
0,Wikipedia,35,2,33,94.3
1,Google Translate,229,17,212,92.6
2,EasyNMT,92,4,88,95.7
3,Lingvanex,100,17,83,83.0


In [32]:
# Comparison: nominal coverage → quality-filtered → search-ready
compare_rows = []
for (_, r1), (_, r2) in zip(tier1_df.iterrows(), tier2_df.iterrows()):
    svc = r1["Service"]
    compare_rows += [
        {"Service": svc, "Stage": "Nominal coverage", "Count": r1["Covered"]},
        {"Service": svc, "Stage": "Quality-filtered", "Count": r1["Usable"]},
        {"Service": svc, "Stage": "Search-ready", "Count": r2["Usable"]},
    ]

compare_df = pd.DataFrame(compare_rows)
stage_order = ["Nominal coverage", "Quality-filtered", "Search-ready"]

funnel = alt.Chart(compare_df).mark_bar().encode(
    x=alt.X("Count:Q", title="Languages"),
    y=alt.Y("Stage:N", sort=stage_order, title=None),
    color=alt.Color(
        "Stage:N",
        sort=stage_order,
        scale=alt.Scale(scheme="blues"),
    ),
    row=alt.Row("Service:N", title=None),
    tooltip=["Service", "Stage", "Count"],
).properties(width=400, height=60, title="Prompt-invariant coverage funnel by filter view")

funnel

alt.Chart(...)